In [255]:
# Phase A: Load and understand the data
import numpy as np
import pandas as pd
import sqlite3

df = pd.read_csv('timesData.csv')
print(df.shape)
print(df.dtypes)
df.head(10)
df.describe()
df.tail(5)
df.isnull().sum()
print(df.tail())


(2603, 14)
world_rank                 object
university_name            object
country                    object
teaching                  float64
international              object
research                  float64
citations                 float64
income                     object
total_score                object
num_students               object
student_staff_ratio       float64
international_students     object
female_male_ratio          object
year                        int64
dtype: object
     world_rank               university_name      country  teaching  \
2598    601-800           Yeungnam University  South Korea      18.6   
2599    601-800   Yıldız Technical University       Turkey      14.5   
2600    601-800      Yokohama City University        Japan      24.0   
2601    601-800  Yokohama National University        Japan      20.1   
2602    601-800            Yuan Ze University       Taiwan      16.2   

     international  research  citations income total_score num_stu

# Data Overview
* 2603 rows (schools) and 14 columns (metrics)
* university_name, country are identifiers
* columns num_students, student_stuff_ratio, international_students are each missing about 2% of the data, and column female_male_ratio misses about 10% of the dataset

In [239]:
df.describe()
df.tail(4)
df['country'].value_counts()[:15]
df['year'].value_counts()
df['country'].value_counts() # 72 countries were represented, and the US leads with 659 results


country
United States of America     659
United Kingdom               300
Germany                      152
Australia                    117
Canada                       108
                            ... 
Unted Kingdom                  1
Cyprus                         1
Unisted States of America      1
Luxembourg                     1
Lithuania                      1
Name: count, Length: 72, dtype: int64

# Descriptive statistics — initial observations
 
* The US dominates by the number of appearance. The data covers 6 years from 2011 to 2016.

* The max values for all metrics seem fine; for the min values, a minimum `student_staff_ratio` near 0.6 is implausibly low and is probably data input error. Besides, `teaching` and `research` both have very low minimum values, suggesting either 
  outlier institutions or additional sentinel values not yet cleaned

In [225]:
# Phase B: Clean the data
print(df[['teaching','research','citations','income','international','total_score']].dtypes) 
df['international'].unique() 

teaching         float64
research         float64
citations        float64
income            object
international     object
total_score       object
dtype: object


array(['72.4', '54.6', '82.3', '29.5', '70.3', '77.7', '77.2', '39.6',
       '90.0', '59.2', '48.1', '62.8', '58.5', '62.4', '93.7', '53.3',
       '-', '90.9', '32.9', '39.1', '91.4', '90.8', '49.0', '49.4',
       '60.5', '18.4', '73.2', '32.6', '64.3', '93.3', '21.5', '31.6',
       '55.9', '97.8', '85.9', '88.0', '68.6', '56.4', '77.9', '67.3',
       '97.4', '44.9', '93.9', '44.5', '43.7', '31.2', '100.0', '66.3',
       '22.1', '23.0', '28.3', '22.6', '43.0', '38.1', '31.8', '52.3',
       '43.1', '35.6', '25.2', '56.5', '64.0', '31.7', '67.2', '16.7',
       '89.6', '42.2', '87.5', '20.9', '84.2', '52.2', '36.7', '72.8',
       '74.2', '66.6', '63.4', '22.5', '65.7', '99.5', '79.1', '92.9',
       '56.8', '69.0', '87.9', '24.4', '87.0', '21.9', '91.3', '35.4',
       '31.0', '37.6', '85.3', '24.2', '85.7', '48.0', '26.8', '62.6',
       '34.1', '83.3', '19.9', '71.8', '47.7', '24.8', '29.2', '34.2',
       '63.0', '95.7', '29.6', '50.2', '91.0', '30.4', '24.7', '73.8',
       '

In [226]:
score_cols = ['teaching', 'international', 'research', 'citations', 'income','total_score']
for score in score_cols:
    df[score]= pd.to_numeric(df[score],errors='coerce') 
    print(df[score].dtype)

print (df.isnull().sum())  # null counts increase as '-' sentinels are coerced to NaN

float64
float64
float64
float64
float64
float64
world_rank                   0
university_name              0
country                      0
teaching                     0
international                9
research                     0
citations                    0
income                     218
total_score               1402
num_students                59
student_staff_ratio         59
international_students      67
female_male_ratio          233
year                         0
dtype: int64


In [241]:
df['rank_numeric'] = df['world_rank'].str.extract(r'(\d+)')
df['rank_numeric'] = pd.to_numeric(df['rank_numeric'], errors='coerce')

df['country'] = df['country'].str.strip() 

df['is_top100'] = (df['rank_numeric'] <= 100).astype(int) # cast bool to int: True to 1, False to 0

In [228]:
# Create a Python dictionary that maps country names to regions. 

df['country'].value_counts()[:10]
region_dic = {'United States of America': 'North America','United Kingdom': 'Europe', 'Germany':'Europe','Australia':'Oceania','Canada':'North America', 'Japan':'Asia',
              'Italy' :'Europe', 'China':'Asia', 'Netherlands':'Europe', 'France':'Europe'}
df['region'] = df['country'].map(region_dic).fillna('Other')
print(df.shape)
print(df.dtypes)

(2603, 17)
world_rank                 object
university_name            object
country                    object
teaching                  float64
international             float64
research                  float64
citations                 float64
income                    float64
total_score               float64
num_students               object
student_staff_ratio       float64
international_students     object
female_male_ratio          object
year                        int64
rank_numeric                int64
is_top100                   int64
region                     object
dtype: object


## Cleaning summary 
* we now have 3 extra columns added (`rank_numeric`, `is_top100`, `region`) for raw data of 2603 rows.
* During the cleaning, we strip off unnecessary spaces in the data and then convert the data types of scores from objects/strings to numerical values by replacing null values with NaNs. 
* `world_rank` ranges like '201–250' and prefixed values like '=5' were resolved to their leading integer and saved in `rank_numeric`.

In [229]:
# Phase C: Explore and aggregate
# aggregate by country: university count, average scores, best rank
country_stats = (
    df.groupby('country').agg(
        numb_of_U=('world_rank','count'),
        avg_total_score  = ('total_score', 'mean'),
        best_rank        = ('rank_numeric', 'min'),
        avg_teaching_score     = ('teaching', 'mean'),
        avg_research_score     = ('research', 'mean'),
        avg_citations_score    = ('citations', 'mean')).sort_values('avg_total_score', ascending=False))
print(country_stats.head(15))

#Part 2: Filter to only rows where is_top100 == 1, then group by country and count. 
# Sort descending: which country has the most top-100 universities?

print(df[df['is_top100'] == 1].groupby('country').size().sort_values(ascending = False)) 

                          numb_of_U  avg_total_score  best_rank  \
country                                                           
Singapore                        12        65.600000         25   
United States of America        659        64.508009          1   
China                            83        61.338889         37   
Switzerland                      47        61.024390          9   
Canada                          108        60.981250         17   
Hong Kong                        34        60.213636         21   
Australia                       117        60.120000         28   
Japan                            98        59.474074         23   
United Kingdom                  300        58.994086          2   
South Korea                      57        58.821739         28   
Sweden                           57        56.584375         28   
Finland                          32        56.033333         76   
Netherlands                      75        55.356522         4

## Country-level observations
* The US has the most top-100 universities by count, but not the highest average total score. Luxembourg leads on average citations score. 
* The above distinction goes to countries with fewer but elite institutions. This reflects a volume vs. concentration distinction.

In [230]:
# average total score by region and year
pd.pivot_table(df, 
               values = 'total_score', 
               index = 'region', 
               columns = 'year',
               aggfunc='mean').round(1)


year,2011,2012,2013,2014,2015,2016
region,,,,,,
Asia,59.4,57.7,63.1,59.5,57.9,68.2
Europe,56.5,53.9,58.6,55.0,56.9,60.3
North America,65.7,63.1,65.4,61.6,62.5,67.2
Oceania,59.8,57.5,62.1,57.7,58.7,64.2
Other,56.1,53.0,57.7,53.7,55.2,58.1


## Regional score trends (2011–2016)

* Scores are broadly stable across years — no region shows a sustained upward or downward trend.

In [231]:
# citations impact vs. global rank
print(df[(df['citations']>90) & (df['rank_numeric']> 50)][['university_name', 'country','rank_numeric','citations']]) 

df[df['research'] > df['teaching']+ 20]
# research-teaching gap
df['research_lead'] = df['research'] - df['teaching']
print(df.sort_values(by = ['research_lead'], ascending= False)[:10])

print(df[df['university_name'].str.contains('MIT')][['total_score','rank_numeric']])

                                        university_name  \
58                                    Boston University   
68                 University of California, Santa Cruz   
72                               University of Adelaide   
74                                       William & Mary   
79                                 University of Sussex   
...                                                 ...   
2001                 Paris Diderot University – Paris 7   
2030               Oregon Health and Science University   
2034  Peter the Great St Petersburg Polytechnic Univ...   
2052                             Wake Forest University   
2069                              University of Iceland   

                       country  rank_numeric  citations  
58    United States of America            59       91.4  
68    United States of America            68       99.6  
72                   Australia            73       90.5  
74    United States of America            75       95.6  
7

In [232]:
# normalize into three relational tables
rankings = df[['university_name', 'year', 'rank_numeric', 'total_score', 'is_top100']].copy() 
scores= df[['university_name', 'year', 'teaching', 'research', 'citations', 'income', 'international']].copy()
institutions = df[['university_name', 'country', 'region', 'num_students', 'student_staff_ratio']].drop_duplicates(subset=['university_name']).copy()

conn = sqlite3.connect('rankings.db')

# load tables into SQLite
rankings.to_sql('rankings', conn, if_exists='replace', index=False)
scores.to_sql('scores', conn, if_exists='replace', index=False)
institutions.to_sql('institutions', conn, if_exists='replace', index=False)

def sql(query): return pd.read_sql_query(query, conn)


## Relational schema
* rankings: the primary keys are university_name, year
* scores: same primary keys
* institutions: the primary key is university_name

In [ ]:
# sql queries and pandas equivalence
# From the rankings table: select university_name, year, rank_numeric, total_score for the most recent year only,
#  where total_score is not null, ordered by total_score descending, limited to 10 rows.
sql("""
    select university_name, year, rank_numeric, total_score
    from rankings
    where year = (SELECT MAX(year) FROM rankings) and total_score is not null                    
    order by total_score DESC
    limit 10
    """)

#pandas equiv:
rankings[(rankings['year']== max(rankings['year'])) & (rankings['total_score'].notna())][['university_name', 'year', 'rank_numeric', 'total_score']].sort_values('total_score', ascending=False) [:10]


# Query 2 — average scores per country; JOIN to get country name; HAVING ≥3 universities
sql("""
    select i.country, avg(s.teaching), avg(s.research), avg(s.citations)
    From scores  s
    Join institutions i
    on s.university_name = i.university_name
    group by i.country
    having count(i.country) >=3
    order by avg(s.research) DESC
    """)

#pandas equiv
scores_insti_merge = pd.merge(scores,institutions, how = "left", on = 'university_name')
scores_insti = (scores_insti_merge.groupby('country').agg( average_teaching = ('teaching','mean'),
                                                         average_research = ('research', 'mean'),
                                                         average_citations = ('citations', 'mean'),
                                                         university_count = ('university_name', 'nunique')
                                                        ))

scores_insti = scores_insti[scores_insti['university_count'] >= 3]
scores_insti.sort_values('average_research', ascending = False)
scores_insti.head()

,average_teaching,average_research,average_citations,university_count
country,,,,
Australia,31.852137,36.118803,55.988889,31
Austria,30.535484,23.535484,57.180645,7
Belgium,35.629730,38.737838,61.062162,7
Brazil,36.308000,27.308000,19.028000,17
Canada,39.007407,42.267593,60.687963,25


In [299]:
# Query 3 — tier classification using CASE WHEN

sql("""
    select 
    case when rank_numeric between 1 and 50 then "Elite"
         when rank_numeric between 51 and 200 then "Top tier"
         when rank_numeric between 201 and 500 then "Strong"
         else "Other" end as tier,
        COUNT(*) AS row_count
    FROM rankings
    GROUP BY tier
    ORDER BY row_count DESC
    """)


# pandas equivalence:pd.cut is the equivalence of case when in sql
rankings['tier']= pd.cut(rankings['rank_numeric'], bins=[0,50,200,500,10000], labels=['Elite','Top tier','Strong','Other'])
print(rankings.groupby('tier').size())



tier
Elite        301
Top tier     900
Strong      1102
Other        300
dtype: int64


/var/folders/9k/ymj25sz10zd4511wlz920s4m0000hf/T/ipykernel_18217/4074805251.py:18: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  print(rankings.groupby('tier').size())


In [235]:
#Query 4 — rank within country via Window function
sql("""
    select r.university_name, i.country, r.rank_numeric, rank() over (partition by i.country order by r.rank_numeric) as "within-country rank"
    from rankings r
    join institutions i
    on r.university_name = i.university_name
    where  r.year = (SELECT max(year) FROM rankings)
    """)

# pandas equiv:
most_recent = rankings[rankings['year'] == rankings['year'].max()]
rank_insti_merge = pd.merge(most_recent, institutions, how="left", on="university_name")
rank_insti_merge['within_country_rank'] = rank_insti_merge.groupby('country')['rank_numeric'].rank(method='min').astype(int)
rank_insti_merge.head()

,university_name,year,rank_numeric,total_score,is_top100,tier,country,region,num_students,student_staff_ratio,within_country_rank
0,California Institute of Technology,2016,1,95.2,1,Elite,United States of America,North America,"2,243",6.9,1
1,University of Oxford,2016,2,94.2,1,Elite,United Kingdom,Europe,"19,919",11.6,1
2,Stanford University,2016,3,93.9,1,Elite,United States of America,North America,"15,596",7.8,2
3,University of Cambridge,2016,4,92.8,1,Elite,United Kingdom,Europe,"18,812",11.8,2
4,Massachusetts Institute of Technology,2016,5,92.0,1,Elite,United States of America,North America,"11,074",9.0,3


In [236]:
# Query 5 universities that improved rank year over year
sql(""" 
    select r_1.university_name, r_1.year, r_2.year, r_1.rank_numeric, r_2.rank_numeric, r_1.rank_numeric-r_2.rank_numeric as "improvement amount" 
    from rankings r_1
    join rankings r_2
    on r_1.university_name = r_2.university_name
    where r_2.year- r_1.year =1 
    order by "improvement amount" desc
    """)
#pandas equiv:
this_year = rankings.copy()
next_year = rankings.copy()
mrankings = this_year.merge(next_year, on ="university_name", suffixes=('_this_year', '_next_year'))
mrankings['improvement'] = mrankings['rank_numeric_this_year']- mrankings['rank_numeric_next_year']
mrankings[(mrankings['year_this_year']-mrankings['year_next_year'] == -1)][['university_name', 'year_this_year', 'year_next_year', 'rank_numeric_this_year', 'rank_numeric_next_year', 'improvement']].sort_values('improvement', ascending=False) 

,university_name,year_this_year,year_next_year,rank_numeric_this_year,rank_numeric_next_year,improvement
9006,University of Erlangen-Nuremberg,2015,2016,276,123,153
9268,Aalborg University,2015,2016,351,201,150
9162,University of Cologne,2015,2016,301,156,145
8875,University of Münster,2015,2016,251,125,126
6516,Middle East Technical University,2014,2015,201,85,116
...,...,...,...,...,...,...
9200,University of Marrakech Cadi Ayyad,2015,2016,301,601,-300
8365,Istanbul Technical University,2015,2016,165,501,-336
8216,Boğaziçi University,2015,2016,139,501,-362
8547,Florida Institute of Technology,2015,2016,200,601,-401


**Extension**: detecting multi-year improvement streaks would require either chaining 
additional self-joins or using a window function with `LAG()` — a natural next step 
for a more advanced version of this analysis.

## SQL–pandas equivalence check
* Results match across all five queries. 
* The main behavioral differences between these two methods are: 
- SQL sorts NULLs last by default while pandas sorts NaN first; 
- SQL JOIN produces no output for unmatched keys but pandas `merge` with `how='left'` preserves all left-side rows.

In [237]:
# Phase E: score matrix statistics 
# per-column summary statistics computed via numpy along axis=0
score_cols = ['teaching', 'research', 'citations', 'income', 'international']
df_clean = df[score_cols].dropna()
X= df_clean.values 

summary = pd.DataFrame({
    'mean': np.mean(X, axis=0),
    'std':  np.std(X, axis=0),
    'p25':  np.percentile(X, 25, axis=0),
    'p75':  np.percentile(X, 75, axis=0),
}, index=score_cols)
summary['IQR'] = summary['p75'] - summary['p25']
print(summary.round(2))

                mean    std    p25    p75    IQR
teaching       37.05  17.24  24.48  45.12  20.65
research       35.40  20.86  19.60  46.12  26.52
citations      60.39  23.01  45.10  78.30  33.20
income         48.97  21.16  33.10  59.00  25.90
international  52.82  22.15  34.30  70.10  35.80



## Score distribution
* `citations` has the highest mean and std, reflecting that a small number of research-intensive institutions score extremely high while most cluster lower.
* `international` has the largest IQR, meaning universities diverge most on this metric.

In [238]:
# center and normalize columns, then compute correlation matrix as Gram matrix

# Center X by subtracting the mean of each column. Shape stays (N, 5).
X_centered = X-X.mean(axis = 0)
X_centered.shape

# Normalize each column by dividing by its standard deviation. 
X_normed = X_centered/ X.std(axis=0)
N = X_normed.shape[0]

# Compute the correlation matrix as (X_normed.T @ X_normed) / N. Call it corr.
corr = (X_normed.T @ X_normed) /N
corr_df = pd.DataFrame(corr, index= score_cols, columns= score_cols)

# Verification: Compare to np.corrcoef(X.T). The maximum absolute difference should be essentially 0.
print((corr_df - pd.DataFrame(np.corrcoef(X.T), index=score_cols, columns=score_cols)).abs().max().max())

print(corr_df.round(3))


6.661338147750939e-16
               teaching  research  citations  income  international
teaching          1.000     0.915      0.489   0.329          0.148
research          0.915     1.000      0.513   0.382          0.257
citations         0.489     0.513      1.000   0.048          0.340
income            0.329     0.382      0.048   1.000          0.011
international     0.148     0.257      0.340   0.011          1.000


# Correlation structure
* `teaching` and `research` are the most correlated pair. Geometrically, this means these two column vectors are nearly parallel, so they 
contribute redundant information to the ranking.
* The correlation matrix here, as the Gram matrix of the mean-centered, column-normalized score vectors, is exactly the matrix PCA eigendecomposes. Its eigenvectors are the principal components; the fact that teaching and research are highly correlated means the first eigenvector loads heavily on both, and the effective dimensionality of the score space is lower than 5.